# 08 · Train efficiently, compare fairly, and resume safely

**Level:** advanced intermediate. **Before this:** implement and explain a training loop.
**Practical outcome:** run a small validation-only learning-rate comparison, then train
with mixed precision, gradient accumulation, clipping, scheduling, early stopping,
and complete resumable checkpoints.

The dataset is Fashion-MNIST and the model is a compact CNN. Short runs make each
mechanism visible; they do not establish optimal hyperparameters. Every component is
shown in notebook cells so you can pause, inspect state, and explain what changes.
An 8 GB NVIDIA GPU is ample for these defaults; CPU fallback disables mixed precision.


## Step 1 · Choose a small, repeatable experiment

Use a fresh kernel and run sequentially. Seeds reduce accidental differences, but
do not promise bitwise-identical results across PyTorch versions or hardware. The
tiny smoke mode tests the pipeline and performs a checkpoint reload without an extra
resumed training epoch. Normal runs perform that extra epoch when patience allows.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project root or in notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the complete course folder; course_utils.py is required.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
import course_utils as cu

DEVICE = cu.configure(seed=42)
SMOKE = cu.SMOKE  # COURSE_SMOKE=1: validation only; too small to judge model quality.
QUICK = True     # Set False for a longer learning experiment, then Restart Kernel and Run All.


In [ ]:
import copy
import random
import time
from itertools import islice
import pandas as pd
from torch import nn

SEED = 42
TUNING_EPOCHS = 1 if SMOKE else (2 if QUICK else 4)
INITIAL_EPOCHS = 1 if SMOKE else (3 if QUICK else 8)
RESUME_EXTRA_EPOCHS = 0 if SMOKE else 1
PATIENCE = 3
MICRO_BATCH_SIZE = 16
ACCUMULATION_STEPS = 4
limits = (64, 32, 32) if SMOKE else ((1600, 400, 400) if QUICK else (10000, 2000, 2000))
train_ds, val_ds, test_ds, class_names = cu.fashion_data(
    train_limit=limits[0], val_limit=limits[1], test_limit=limits[2], seed=SEED,
)
val_dl = cu.loader(val_ds, batch_size=64)
test_dl = cu.loader(test_ds, batch_size=64)
print("Train / validation / test:", len(train_ds), len(val_ds), len(test_ds))
print("Nominal effective batch:", MICRO_BATCH_SIZE * ACCUMULATION_STEPS)


## Step 2 · Compare two choices using validation only

Keep architecture, split, initialization seed, batch size, and epoch budget fixed.
Rebuild both the model and shuffled loader for each candidate. The table records the
validation result of each run's best epoch, because `fit` restores that state.
This is a small teaching search, not evidence that one learning rate is universally
better. A serious study repeats multiple seeds and reports variability.


In [ ]:
candidates = [
    {"lr": 1e-3, "weight_decay": 1e-4},
    {"lr": 3e-3, "weight_decay": 1e-4},
]
trial_rows, trial_histories = [], {}
for trial_id, config in enumerate(candidates):
    cu.seed_everything(SEED)
    candidate_model = cu.SmallCNN(num_classes=len(class_names), in_channels=1).to(DEVICE)
    candidate_train_dl = cu.loader(train_ds, batch_size=64, shuffle=True, seed=SEED)
    history = cu.fit(candidate_model, candidate_train_dl, val_dl,
                     epochs=TUNING_EPOCHS, device=DEVICE, **config)
    metrics = cu.evaluate(candidate_model, val_dl, DEVICE)
    trial_rows.append({"trial": trial_id, **config,
                       "val_loss": metrics["loss"], "val_accuracy": metrics["accuracy"]})
    trial_histories[trial_id] = history
    del candidate_model

trial_results = pd.DataFrame(trial_rows).sort_values(["val_loss", "trial"])
display(trial_results)
chosen_id = int(trial_results.iloc[0]["trial"])
chosen_config = dict(candidates[chosen_id])
print("Selected using validation loss:", chosen_config)


## Step 3 · Read the curves before adding machinery

Plot validation loss for both learning rates. A lower final loss is not the same as
the best epoch's loss. Large oscillations, divergence, and widening train-validation
gaps answer different questions. If you alter settings after inspecting the curves,
record the new validation experiment and leave the test set untouched.


In [ ]:
for trial_id, history in trial_histories.items():
    plt.plot([row["epoch"] for row in history], [row["val_loss"] for row in history],
             marker="o", label=f"lr={candidates[trial_id]['lr']}")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Same initialization, split, and budget")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## Step 4 · Understand the optimization recipe

| Mechanism | What it changes | What to watch |
|---|---|---|
| CUDA autocast + gradient scaling | Some operations use lower precision | Numerical stability and actual measured speed |
| Gradient accumulation | Several small forward/backward passes make one update | Correct normalization of the final partial window |
| Gradient clipping | Limits the unscaled gradient norm | Clip only after accumulation and unscaling |
| Learning-rate scheduler | Changes the rate between epochs | Save and resume its state |
| Early stopping | Stops after repeated validation non-improvement | Restore the best checkpoint for final inference |

We start a **new** run with the selected learning rate. Accumulation and mixed
precision change the training procedure, so do not present this run as a controlled
speed-only comparison with Step 2. BatchNorm and stochastic layers can also make
microbatch training differ from one large physical batch.


In [ ]:
AMP_ENABLED = DEVICE.type == "cuda"
run_config = {
    **chosen_config, "micro_batch_size": MICRO_BATCH_SIZE,
    "accumulation_steps": ACCUMULATION_STEPS,
    "amp_enabled": AMP_ENABLED, "seed": SEED, "patience": PATIENCE,
    "max_grad_norm": 1.0,
}

def make_components(config):
    cu.seed_everything(config["seed"])
    network = cu.SmallCNN(num_classes=len(class_names), in_channels=1).to(DEVICE)
    optimizer = torch.optim.AdamW(network.parameters(), lr=config["lr"],
                                  weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
    scaler = torch.amp.GradScaler("cuda", enabled=config["amp_enabled"])
    return network, optimizer, scheduler, scaler

model, optimizer, scheduler, scaler = make_components(run_config)
print("CUDA mixed precision:", AMP_ENABLED)
print("Starting learning rate:", optimizer.param_groups[0]["lr"])


## Step 5 · Accumulate a sample-weighted effective batch

A final accumulation window may have fewer microbatches, and its last batch may have
fewer samples. Dividing every mean loss by a fixed accumulation count underweights
that window. For a window of `N` samples, each microbatch mean gets weight `n / N`.
The accumulated gradient is then the gradient of the mean loss across that window.

Only one microbatch is moved to the GPU at a time. Keep gradients scaled until the
whole window is complete, then unscale once, clip, step, update the scaler, and clear
gradients. Sample-weighted normalization does not make BatchNorm or dropout exactly
equivalent to processing a large physical batch.


In [ ]:
def optimized_epoch(network, dataloader, optimizer, scaler, config):
    network.train()
    total_loss, correct, count, updates = 0.0, 0, 0, 0
    iterator = iter(dataloader)
    while True:
        # These are CPU batches; GPU activations exist for just one at a time.
        window = list(islice(iterator, config["accumulation_steps"]))
        if not window:
            break
        window_samples = sum(labels.numel() for _, labels in window)
        optimizer.zero_grad(set_to_none=True)
        for images, labels in window:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            n = labels.numel()
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16,
                                enabled=config["amp_enabled"]):
                logits = network(images)
                mean_loss = nn.functional.cross_entropy(logits, labels)
                contribution = mean_loss * (n / window_samples)
            scaler.scale(contribution).backward()
            total_loss += mean_loss.detach().item() * n
            correct += (logits.detach().argmax(1) == labels).sum().item()
            count += n

        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(network.parameters(), config["max_grad_norm"])
        old_scale = scaler.get_scale()
        scaler.step(optimizer)
        scaler.update()
        # A reduced scale indicates overflow and an optimizer update that was skipped.
        if scaler.get_scale() >= old_scale:
            updates += 1
        optimizer.zero_grad(set_to_none=True)
    return {"loss": total_loss / count, "accuracy": correct / count, "updates": updates}


## Step 6 · Check the partial-window arithmetic directly

This tiny linear-model check compares a physical batch of eleven examples with
microbatches of sizes 4, 4, and 3. It isolates the normalization rule without dropout,
BatchNorm, or mixed-precision rounding. Both gradients should agree within tolerance.
This is a mechanics check, not a model-accuracy test.


In [ ]:
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(123)
    reference = nn.Linear(5, 3)
    accumulated = copy.deepcopy(reference)
    x_demo, y_demo = torch.randn(11, 5), torch.randint(0, 3, (11,))
    nn.functional.cross_entropy(reference(x_demo), y_demo).backward()
    for start, end in [(0, 4), (4, 8), (8, 11)]:
        mean_loss = nn.functional.cross_entropy(accumulated(x_demo[start:end]), y_demo[start:end])
        (mean_loss * ((end - start) / len(y_demo))).backward()
    for full_parameter, micro_parameter in zip(reference.parameters(), accumulated.parameters()):
        torch.testing.assert_close(full_parameter.grad, micro_parameter.grad, rtol=1e-5, atol=1e-7)
print("Passed: partial-window gradients match a physical batch for this deterministic model.")


## Step 7 · Capture the states needed to resume

Inference needs model weights and preprocessing. Training continuation also needs
optimizer momentum, scheduler position, AMP scale, epoch, stopping counters, history,
and random-generator states. The helper stores NumPy's RNG array as a tensor so the
checkpoint can be reloaded with `weights_only=True`. This notebook saves only its
own checkpoints and uses no arbitrary-object unpickling.

We create the shuffled loader from `seed + epoch` for each epoch. Thus epoch-boundary
resume recovers its sample order without storing a partially consumed iterator.
Mid-epoch resume and arbitrary worker-prefetch state are outside this demonstration.


In [ ]:
LAST_PATH = cu.ARTIFACTS / "08_last_checkpoint.pt"
BEST_PATH = cu.ARTIFACTS / "08_best_checkpoint.pt"

def capture_rng():
    numpy_state = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": {
            "name": numpy_state[0],
            "keys": torch.tensor(numpy_state[1].astype(np.int64)),
            "position": int(numpy_state[2]),
            "has_gauss": int(numpy_state[3]), "cached_gaussian": float(numpy_state[4]),
        },
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else [],
    }

def restore_rng(state):
    random.setstate(state["python"])
    numpy_state = state["numpy"]
    np.random.set_state((numpy_state["name"], numpy_state["keys"].cpu().numpy().astype(np.uint32),
                         numpy_state["position"], numpy_state["has_gauss"], numpy_state["cached_gaussian"]))
    torch.set_rng_state(state["torch"].cpu())
    if DEVICE.type == "cuda" and state["cuda"]:
        torch.cuda.set_rng_state_all([rng.cpu() for rng in state["cuda"]])

def checkpoint_payload(network, optimizer, scheduler, scaler, training_state, config):
    return {
        "model_state": network.state_dict(), "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(), "scaler_state": scaler.state_dict(),
        "training_state": training_state, "config": config, "rng": capture_rng(),
        "class_names": list(class_names), "architecture": "SmallCNN", "in_channels": 1,
        "preprocessing": "FashionMNIST grayscale 28x28, ToTensor(), no normalization",
    }


## Step 8 · Schedule, validate, save, and stop

Validation uses full precision and no gradients. Step the scheduler after an epoch
with an actual optimizer update; save its updated state. `last` is for continuing
training, while `best` is for inference. The best checkpoint may precede the most
recent checkpoint. Early stopping tracks **validation loss**, not training loss.

CUDA kernels are asynchronous, so synchronize before starting and stopping the
wall-clock timer. The displayed epoch time includes training and validation but
excludes checkpoint writing; state that scope when reporting a measurement.


In [ ]:
def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def train_until(network, optimizer, scheduler, scaler, state, config, stop_epoch):
    for epoch in range(state["next_epoch"], stop_epoch):
        if state["bad_epochs"] >= config["patience"]:
            print("Patience already exhausted; keeping the best validation checkpoint.")
            break
        train_dl = cu.loader(train_ds, batch_size=config["micro_batch_size"],
                             shuffle=True, seed=config["seed"] + epoch)
        current_lr = optimizer.param_groups[0]["lr"]
        synchronize()
        start = time.perf_counter()
        train_metrics = optimized_epoch(network, train_dl, optimizer, scaler, config)
        val_metrics = cu.evaluate(network, val_dl, DEVICE)
        synchronize()
        elapsed = time.perf_counter() - start
        if train_metrics["updates"]:
            scheduler.step()

        improved = val_metrics["loss"] < state["best_loss"]
        if improved:
            state["best_loss"] = val_metrics["loss"]
            state["bad_epochs"] = 0
        else:
            state["bad_epochs"] += 1
        state["next_epoch"] = epoch + 1
        state["history"].append({
            "epoch": epoch + 1, "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"], "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"], "lr": current_lr,
            "seconds": elapsed, "optimizer_updates": train_metrics["updates"],
        })
        payload = checkpoint_payload(network, optimizer, scheduler, scaler, state, config)
        torch.save(payload, LAST_PATH)
        if improved:
            torch.save(payload, BEST_PATH)
        print(f"Epoch {epoch + 1}: val_loss={val_metrics['loss']:.4f}, "
              f"lr={current_lr:.2g}, seconds={elapsed:.2f}, "
              f"bad_epochs={state['bad_epochs']}")
        if state["bad_epochs"] >= config["patience"]:
            print("Early stopping triggered.")
            break
    return state

state = {"next_epoch": 0, "best_loss": float("inf"), "bad_epochs": 0, "history": []}
state = train_until(model, optimizer, scheduler, scaler, state, run_config, INITIAL_EPOCHS)
print("Last checkpoint:", LAST_PATH)
print("Best checkpoint:", BEST_PATH)


## Step 9 · Simulate an interruption and restore all components

Build new objects, load their saved states, and restore random state **after** object
construction (initialization consumes random numbers). For this teaching run, resume
on the same hardware and with unchanged data/configuration. Moving between CPU and
CUDA requires a deliberate AMP policy change and is not an exact continuation.
The assertions verify that reload reproduces the saved weights and scheduling state.


In [ ]:
checkpoint = torch.load(LAST_PATH, map_location=DEVICE, weights_only=True)
resume_config = checkpoint["config"]
assert resume_config["amp_enabled"] == (DEVICE.type == "cuda"), "Resume on the same device type."
resumed_model, resumed_optimizer, resumed_scheduler, resumed_scaler = make_components(resume_config)
resumed_model.load_state_dict(checkpoint["model_state"])
resumed_optimizer.load_state_dict(checkpoint["optimizer_state"])
resumed_scheduler.load_state_dict(checkpoint["scheduler_state"])
resumed_scaler.load_state_dict(checkpoint["scaler_state"])
resumed_state = checkpoint["training_state"]
restore_rng(checkpoint["rng"])

for name, tensor in model.state_dict().items():
    torch.testing.assert_close(tensor, resumed_model.state_dict()[name], rtol=0, atol=0)
assert resumed_scheduler.state_dict() == scheduler.state_dict()
assert resumed_scaler.state_dict() == scaler.state_dict()
assert resumed_optimizer.param_groups[0]["lr"] == optimizer.param_groups[0]["lr"]
print("Restored next epoch:", resumed_state["next_epoch"] + 1)
print("Restored LR:", resumed_optimizer.param_groups[0]["lr"])
print("Restored bad-epoch counter:", resumed_state["bad_epochs"])
del model, optimizer, scheduler, scaler  # Release the original run's GPU objects.


## Step 10 · Continue without resetting patience or the schedule

The next call adds one epoch in a normal run, provided patience has not already been
exhausted. It preserves history and counters. If early stopping already fired, the
cell reports that and performs no further optimization. Smoke mode reloads only.
Do not restart from the best weights while retaining the last optimizer state:
those states come from different points in the training trajectory.


In [ ]:
resume_stop = resumed_state["next_epoch"] + RESUME_EXTRA_EPOCHS
resumed_state = train_until(resumed_model, resumed_optimizer, resumed_scheduler, resumed_scaler,
                            resumed_state, resume_config, resume_stop)
history_table = pd.DataFrame(resumed_state["history"])
display(history_table)
cu.plot_history(resumed_state["history"])
history_table.plot(x="epoch", y="lr", marker="o", legend=False, title="Learning rate used each epoch")
plt.ylabel("Learning rate")
plt.show()


## Step 11 · Restore the best validation model and evaluate test once

The final inference model comes from `best`, not automatically from the last epoch.
The test set has not influenced the learning-rate comparison, early stopping, or
checkpoint selection. Its small teaching subset has substantial sampling uncertainty.
Do not interpret one short-run score as a production performance guarantee.


In [ ]:
best_checkpoint = torch.load(BEST_PATH, map_location=DEVICE, weights_only=True)
resumed_model.load_state_dict(best_checkpoint["model_state"])
resumed_model.eval()
print("Best epoch:", best_checkpoint["training_state"]["next_epoch"])
print("Best validation loss:", best_checkpoint["training_state"]["best_loss"])
print("Held-out test:", cu.evaluate(resumed_model, test_dl, DEVICE))


## Step 12 · Measure inference instead of assuming a speedup

Warm up, then time repeated forward passes on one already-transferred batch.
Synchronize CUDA around the measurement. These numbers measure **model forward time**,
excluding image decoding, disk I/O, data transfer, and application overhead. Compare
FP32 and AMP on your actual GPU; a small CNN can be too small to benefit from AMP.
No result is prefilled, and a speedup is not guaranteed.


In [ ]:
benchmark_images, _ = next(iter(val_dl))
benchmark_images = benchmark_images.to(DEVICE)

def benchmark_forward(network, images, use_amp, repeats):
    network.eval()
    with torch.inference_mode():
        for _ in range(3):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                _ = network(images)
        synchronize()
        start = time.perf_counter()
        for _ in range(repeats):
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                _ = network(images)
        synchronize()
    return 1000 * (time.perf_counter() - start) / repeats

repeats = 2 if SMOKE else 30
timings = [{"mode": "FP32", "ms_per_batch": benchmark_forward(resumed_model, benchmark_images, False, repeats)}]
if AMP_ENABLED:
    timings.append({"mode": "CUDA AMP", "ms_per_batch": benchmark_forward(resumed_model, benchmark_images, True, repeats)})
display(pd.DataFrame(timings))
print("Batch size:", len(benchmark_images), "| Device:", DEVICE)
if DEVICE.type == "cuda":
    print("Current allocated GPU memory (MiB):", round(torch.cuda.memory_allocated() / 2**20, 1))


## Step 13 · Demonstration challenges

1. Set `MICRO_BATCH_SIZE=15` and `ACCUMULATION_STEPS=3`. Explain the final window's
   weights, then rerun the training procedure as a new validation experiment.
2. Print `scaler.get_scale()` during a CUDA run. Why must the scale remain fixed
   while accumulating one effective batch?
3. Inspect the checkpoint keys. Which states would be missing if you saved only
   `model.state_dict()`?
4. Vary the patience and scheduler period in a new run. Record why each change was
   made and keep the final test set outside this process.

<details><summary>Hints / solutions</summary>

Use each microbatch's sample count divided by the window's actual sample count.
Mixing gradients created with different scale factors makes their sum inconsistent.
Before clipping, unscale once after the full window; update the scaler after the
optimizer step. Resume needs optimizer, scheduler, scaler, random state, epoch,
history, class/preprocessing information, and early-stopping counters.

```python
print(checkpoint.keys())
print(checkpoint["training_state"]["next_epoch"])
print(checkpoint["scheduler_state"])
```

For a stronger continuation check, compare an uninterrupted deterministic CPU run
against an epoch-boundary save/reload run with the same data and epoch seeds. Full
cross-hardware bitwise identity is not promised by this notebook.

</details>

**References:** [PyTorch AMP examples: accumulation and clipping](https://docs.pytorch.org/docs/stable/notes/amp_examples.html),
[AMP performance recipe](https://docs.pytorch.org/tutorials/recipes/recipes/amp_recipe.html),
[Saving a general checkpoint](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html),
[Reproducibility notes](https://docs.pytorch.org/docs/stable/notes/randomness.html).
